# Staged Transaction Linking Experiment (v2 — CSV I/O)

**Goal**: Two-stage receipt-to-bank-statement matching that processes each image independently.

Unlike the single-pass approach (which sends receipt + bank statement together),
this experiment splits inference into two stages:

- **Stage 1** (receipt image only): Extract store name, date, and total for each receipt
- **Stage 2** (bank statement image only): Dynamic prompt with Stage 1 results as text,
  find matching debit transactions

**v2 changes** (vs `staged_transaction_linking.ipynb`):
- Ground truth loaded from CSV (`evaluation_data/transaction_link_ground_truth.csv`)
- Results written to CSV (`evaluation_data/transaction_link_model_results.csv`)
- Both CSVs compatible with `evaluate_model.ipynb`

In [ ]:
"""Cell 1: Imports and sys.path setup."""

import csv
import re
import sys
from pathlib import Path

import torch
import yaml
from IPython.display import display
from PIL import Image

# Add project root to path so we can import from models/, common/, etc.
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from common.evaluation_metrics import load_ground_truth  # noqa: E402
from common.pipeline_config import PipelineConfig  # noqa: E402
from common.simple_prompt_loader import SimplePromptLoader  # noqa: E402
from common.unified_bank_extractor import (  # noqa: E402
    ColumnMapping,
    ColumnMatcher,
    ResponseParser,
)
from models.internvl3_image_preprocessor import InternVL3ImagePreprocessor  # noqa: E402
from models.registry import get_model  # noqa: E402

print(f"Project root: {PROJECT_ROOT}")
print(f"PyTorch: {torch.__version__}, CUDA: {torch.cuda.is_available()}")

In [ ]:
"""Cell 2: Load experiment config and ground truth from CSV."""

config_path = PROJECT_ROOT / "config" / "experiment_config.yml"
with config_path.open() as f:
    exp_config = yaml.safe_load(f)

# Base settings
model_type = exp_config["model"]["type"]
max_tiles = exp_config["model"]["max_tiles"]
max_new_tokens = exp_config["model"]["max_new_tokens"]

# Staged multi-receipt overrides
staged = exp_config["staged_multi_receipt"]
max_tiles = staged["model"].get("max_tiles", max_tiles)
max_new_tokens = staged["model"].get("max_new_tokens", max_new_tokens)
prompt_file = staged["prompts"]["file"]
stage1_key = staged["prompts"]["stage1_key"]
column_extraction_key = staged["prompts"]["column_extraction_key"]
stage2_key = staged["prompts"]["stage2_key"]

# Ground truth from CSV (not YAML)
gt_csv_path = PROJECT_ROOT / "evaluation_data" / "transaction_link_ground_truth.csv"
ground_truth = load_ground_truth(str(gt_csv_path))
print(f"Ground truth images: {list(ground_truth.keys())}")

# Stage-specific token limits from prompt YAML settings
prompt_settings = SimplePromptLoader.get_settings(prompt_file)
stage1_max_tokens = prompt_settings.get("max_new_tokens_stage1", 500)
stage2a_max_tokens = prompt_settings.get("max_new_tokens_stage2a", 200)
stage2_max_tokens = prompt_settings.get("max_new_tokens_stage2", 1500)

# Data dir (used by PipelineConfig for model loading)
data_dir = PROJECT_ROOT / exp_config["data"]["dir"]

# Results CSV output path
results_csv_path = (
    PROJECT_ROOT / "evaluation_data" / "transaction_link_model_results.csv"
)

print(f"Model: {model_type}, max_tiles: {max_tiles}")
print(f"Prompt file: {prompt_file}")
print(f"Stage 1: key={stage1_key}, max_tokens={stage1_max_tokens}")
print(f"Stage 2a: key={column_extraction_key}, max_tokens={stage2a_max_tokens}")
print(f"Stage 2b: key={stage2_key}, max_tokens={stage2_max_tokens}")
print(f"Ground truth entries: {len(ground_truth)}")
print(f"Results CSV: {results_csv_path}")

In [ ]:
"""Cell 3: Load multi-receipt image from evaluation_data."""

eval_data_dir = PROJECT_ROOT / "evaluation_data"
receipt_filename = "synthetic_multi_receipt_page.png"
receipt_path = eval_data_dir / receipt_filename

if not receipt_path.exists():
    raise FileNotFoundError(
        f"Receipt image not found: {receipt_path}\n"
        "Run: python -c 'from experiments.synthetic.generate_multi_receipt_page import "
        'generate_multi_receipt_page; generate_multi_receipt_page(Path("evaluation_data/synthetic_multi_receipt_page.png"))\''
    )

print(f"Loading receipt image: {receipt_path}")
receipt_img = Image.open(receipt_path)
display(receipt_img)

In [ ]:
"""Cell 4: Load bank statement image from evaluation_data."""

statement_path = eval_data_dir / "synthetic_bank_statement.png"

if not statement_path.exists():
    raise FileNotFoundError(
        f"Bank statement image not found: {statement_path}\n"
        "Run: python -c 'from experiments.synthetic.generate_bank_statement import "
        'generate_bank_statement; generate_bank_statement(Path("evaluation_data/synthetic_bank_statement.png"))\''
    )

print(f"Loading bank statement: {statement_path}")
statement_img = Image.open(statement_path)
display(statement_img)

In [ ]:
"""Cell 5: Load InternVL3 model via registry."""

run_config_path = PROJECT_ROOT / "config" / "run_config.yml"
with run_config_path.open() as f:
    run_config = yaml.safe_load(f)

# Resolve model path
model_path = run_config.get("model", {}).get("path")
if not model_path:
    model_path = (
        run_config.get("model_loading", {}).get("default_paths", {}).get(model_type)
    )
if not model_path:
    raise FileNotFoundError(
        f"No model path found for '{model_type}' in run_config.yml. "
        "Set model.path or model_loading.default_paths.{model_type}."
    )
model_path = Path(model_path)
print(f"Model path: {model_path}")

cfg = PipelineConfig(
    data_dir=data_dir,
    output_dir=data_dir,
    model_path=model_path,
    model_type=model_type,
    max_tiles=max_tiles,
    flash_attn=exp_config["model"]["flash_attn"],
    dtype=exp_config["model"]["dtype"],
    max_new_tokens=max_new_tokens,
)

registration = get_model(model_type)
model_ctx = registration.loader(cfg)
model, tokenizer = model_ctx.__enter__()
print(f"Model loaded: {type(model).__name__}")
print(f"Device: {next(model.parameters()).device}")

In [ ]:
"""Cell 6: Stage 1 — Preprocess receipt image ONLY (no bank statement)."""

preprocessor = InternVL3ImagePreprocessor(max_tiles=max_tiles)

receipt_pv = preprocessor.load_image(str(receipt_path), model, max_num=max_tiles)
receipt_patches = [receipt_pv.shape[0]]

print(f"Receipt tiles: {receipt_pv.shape[0]}")
print(f"Receipt pv shape: {receipt_pv.shape}")
print("(Bank statement NOT loaded yet — Stage 1 uses receipt only)")

In [ ]:
"""Cell 7: Stage 1 — Load prompt and run inference on receipt."""

stage1_prompt = SimplePromptLoader.load_prompt(prompt_file, stage1_key)
stage1_question = f"<image>\n{stage1_prompt}"

print(f"Stage 1 prompt ({len(stage1_prompt)} chars):")
print("=" * 60)
print(stage1_prompt)
print("=" * 60)

stage1_gen_config = {
    "max_new_tokens": stage1_max_tokens,
    "do_sample": False,
    "num_beams": 1,
}

print("\nRunning Stage 1 inference (receipt only)...")
stage1_response = model.chat(
    tokenizer,
    receipt_pv,
    stage1_question,
    stage1_gen_config,
    num_patches_list=receipt_patches,
)
print("Stage 1 inference complete.")

In [ ]:
"""Cell 8: Stage 1 — Display and parse results (STORE/DATE/TOTAL)."""


def parse_stage1_response(text: str) -> list[dict[str, str]]:
    """Parse Stage 1 response into list of {STORE, DATE, TOTAL} dicts.

    Handles both delimited and markdown header formats:
      --- RECEIPT 1 ---
      #### Receipt 1
    """
    sections = re.split(r"---\s*RECEIPT\s+\d+\s*---", text)
    if len(sections) <= 1:
        sections = re.split(r"#{2,4}\s+Receipt\s+\d+", text)
    results = []
    for section in sections[1:]:
        entry: dict[str, str] = {}
        for line in section.splitlines():
            # Markdown bold: - **STORE:** value  or  **STORE:** value
            md_match = re.match(r"^[-\s]*\*\*([A-Z_]+):\*\*\s*(.+)$", line.strip())
            if md_match:
                entry[md_match.group(1)] = md_match.group(2).strip()
                continue
            # Plain: STORE: value
            plain_match = re.match(r"^([A-Z_]+):\s*(.+)$", line.strip())
            if plain_match:
                entry[plain_match.group(1)] = plain_match.group(2).strip()
        if entry:
            results.append(entry)
    return results


print("Stage 1 raw response:")
print("=" * 60)
print(stage1_response)
print("=" * 60)

stage1_receipts = parse_stage1_response(stage1_response)
print(f"\nParsed {len(stage1_receipts)} receipts:")
for i, r in enumerate(stage1_receipts, 1):
    print(
        f"  Receipt {i}: {r.get('STORE', '?')} | {r.get('DATE', '?')} | {r.get('TOTAL', '?')}"
    )

In [ ]:
"""Cell 9: Build purchases text from Stage 1 results for injection into Stage 2b."""

purchases_lines = []
for i, r in enumerate(stage1_receipts, 1):
    purchases_lines.append(
        f"Purchase {i}: Store={r.get('STORE', 'UNKNOWN')}, "
        f"Date={r.get('DATE', 'UNKNOWN')}, "
        f"Total={r.get('TOTAL', 'UNKNOWN')}"
    )
purchases_text = "\n".join(purchases_lines)

print("Purchases text for Stage 2b:")
print(purchases_text)

In [ ]:
"""Cell 10: Stage 2 — Preprocess bank statement image ONLY."""

statement_pv = preprocessor.load_image(str(statement_path), model, max_num=max_tiles)
statement_patches = [statement_pv.shape[0]]

print(f"Statement tiles: {statement_pv.shape[0]}")
print(f"Statement pv shape: {statement_pv.shape}")
print("(Receipt pixel_values NOT included — Stage 2 uses statement only)")

In [ ]:
"""Cell 11: Stage 2a — Extract column headers from bank statement."""

stage2a_prompt = SimplePromptLoader.load_prompt(prompt_file, column_extraction_key)
stage2a_question = f"<image>\n{stage2a_prompt}"

print(f"Stage 2a prompt ({len(stage2a_prompt)} chars):")
print("=" * 60)
print(stage2a_prompt)
print("=" * 60)

stage2a_gen_config = {
    "max_new_tokens": stage2a_max_tokens,
    "do_sample": False,
    "num_beams": 1,
}

print("\nRunning Stage 2a inference (bank statement — column extraction)...")
stage2a_response = model.chat(
    tokenizer,
    statement_pv,
    stage2a_question,
    stage2a_gen_config,
    num_patches_list=statement_patches,
)
print("Stage 2a inference complete.")

print("\nExtracted column headers:")
print("=" * 60)
print(stage2a_response)
print("=" * 60)

In [ ]:
"""Cell 12: Map column headers to semantic roles for Stage 2b.

Uses ColumnMatcher + bank_column_patterns.yaml to map raw headers
(e.g. "Trans Date", "Particulars", "Withdrawals") to semantic roles
(date, description, debit) so Stage 2b references exact column names.
"""

# Parse the numbered list from Stage 2a into a list of header strings
headers = ResponseParser.parse_headers(stage2a_response)
print(f"Parsed headers: {headers}")

# Match headers to semantic roles using bank_column_patterns.yaml
column_mapping: ColumnMapping = ColumnMatcher().match(headers)
print("\nColumn mapping:")
print(f"  date:        {column_mapping.date}")
print(f"  description: {column_mapping.description}")
print(f"  debit:       {column_mapping.debit}")
print(f"  credit:      {column_mapping.credit}")
print(f"  balance:     {column_mapping.balance}")

# Extract the 3 required columns — fail explicitly if any is missing
date_column = column_mapping.date
description_column = column_mapping.description
debit_column = column_mapping.debit

missing = []
if date_column is None:
    missing.append("date")
if description_column is None:
    missing.append("description")
if debit_column is None:
    missing.append("debit")

if missing:
    raise ValueError(
        f"Column mapping failed for required roles: {missing}. "
        f"Detected headers: {headers}. "
        f"Check config/bank_column_patterns.yaml keywords."
    )

print("\nStage 2b will use:")
print(f"  date_column        = '{date_column}'")
print(f"  description_column = '{description_column}'")
print(f"  debit_column       = '{debit_column}'")

In [ ]:
"""Cell 13: Stage 2b — Build dynamic prompt and run inference on bank statement."""

# Load Stage 2b template and inject purchases + mapped column names
stage2_template = SimplePromptLoader.load_prompt(prompt_file, stage2_key)
stage2_prompt = stage2_template.format(
    purchases_text=purchases_text,
    date_column=date_column,
    description_column=description_column,
    debit_column=debit_column,
)

print(f"Stage 2b prompt ({len(stage2_prompt)} chars):")
print("=" * 60)
print(stage2_prompt)
print("=" * 60)

stage2_question = f"<image>\n{stage2_prompt}"

stage2_gen_config = {
    "max_new_tokens": stage2_max_tokens,
    "do_sample": False,
    "num_beams": 1,
}

print("\nRunning Stage 2b inference (bank statement — transaction matching)...")
stage2_response = model.chat(
    tokenizer,
    statement_pv,
    stage2_question,
    stage2_gen_config,
    num_patches_list=statement_patches,
)
print("Stage 2b inference complete.")

In [ ]:
"""Cell 14: Stage 2b — Display and parse matching results."""


def parse_kv_response(text: str) -> dict[str, str]:
    """Extract KEY: VALUE pairs from model response."""
    results: dict[str, str] = {}
    for line in text.splitlines():
        md_match = re.match(r"^[-\s]*\*\*([A-Z_]+):\*\*\s*(.+)$", line.strip())
        if md_match:
            results[md_match.group(1)] = md_match.group(2).strip()
            continue
        plain_match = re.match(r"^([A-Z_]+):\s*(.*)$", line.strip())
        if plain_match:
            value = plain_match.group(2).strip()
            results[plain_match.group(1)] = value if value else "NOT_FOUND"
    return results


def parse_multi_receipt_response(text: str) -> list[dict[str, str]]:
    """Split response on receipt headers and parse each section."""
    sections = re.split(r"---\s*RECEIPT\s+\d+\s*---", text)
    if len(sections) <= 1:
        sections = re.split(r"#{2,4}\s+Receipt\s+\d+", text)
    results = []
    for section in sections[1:]:
        parsed = parse_kv_response(section)
        if parsed:
            results.append(parsed)
    return results


print("Stage 2b raw response:")
print("=" * 60)
print(stage2_response)
print("=" * 60)

parsed_receipts = parse_multi_receipt_response(stage2_response)
print(f"\nParsed {len(parsed_receipts)} receipt matches:")
for i, p in enumerate(parsed_receipts, 1):
    print(f"\n--- Receipt {i} ---")
    for key, value in p.items():
        print(f"  {key}: {value}")

In [ ]:
"""Cell 15: Build results dict and write results CSV.

Maps Stage 1 + Stage 2b output to standard CSV field names:
  RECEIPT_DATE, RECEIPT_DESCRIPTION, RECEIPT_TOTAL,
  BANK_TRANSACTION_DATE, BANK_TRANSACTION_DESCRIPTION, BANK_TRANSACTION_DEBIT

Multi-receipt values are pipe-joined (" | ") with positional alignment.
If a receipt has no bank match (MATCHED_TRANSACTION != FOUND), its
bank-side fields are NOT_FOUND.
"""

# Determine image filename (just the basename, not full path)
image_filename = receipt_path.name

# Build per-receipt field lists
receipt_dates = []
receipt_descriptions = []
receipt_totals = []
bank_dates = []
bank_descriptions = []
bank_debits = []

for i, parsed_r in enumerate(parsed_receipts):
    # Receipt-side fields from Stage 1 (fall back to Stage 2b parsed output)
    s1 = stage1_receipts[i] if i < len(stage1_receipts) else {}
    receipt_dates.append(s1.get("DATE", parsed_r.get("RECEIPT_DATE", "NOT_FOUND")))
    receipt_descriptions.append(
        s1.get("STORE", parsed_r.get("RECEIPT_STORE", "NOT_FOUND"))
    )
    receipt_totals.append(s1.get("TOTAL", parsed_r.get("RECEIPT_TOTAL", "NOT_FOUND")))

    # Bank-side fields from Stage 2b (only if matched)
    matched = parsed_r.get("MATCHED_TRANSACTION", "NOT_FOUND").upper() == "FOUND"
    if matched:
        bank_dates.append(parsed_r.get("TRANSACTION_DATE", "NOT_FOUND"))
        bank_descriptions.append(parsed_r.get("TRANSACTION_DESCRIPTION", "NOT_FOUND"))
        bank_debits.append(parsed_r.get("TRANSACTION_AMOUNT", "NOT_FOUND"))
    else:
        bank_dates.append("NOT_FOUND")
        bank_descriptions.append("NOT_FOUND")
        bank_debits.append("NOT_FOUND")

PIPE = " | "
results_row = {
    "image_file": image_filename,
    "DOCUMENT_TYPE": "TRANSACTION_LINK",
    "RECEIPT_DATE": PIPE.join(receipt_dates),
    "RECEIPT_DESCRIPTION": PIPE.join(receipt_descriptions),
    "RECEIPT_TOTAL": PIPE.join(receipt_totals),
    "BANK_TRANSACTION_DATE": PIPE.join(bank_dates),
    "BANK_TRANSACTION_DESCRIPTION": PIPE.join(bank_descriptions),
    "BANK_TRANSACTION_DEBIT": PIPE.join(bank_debits),
}

print("Results row:")
for k, v in results_row.items():
    print(f"  {k}: {v}")

# Write CSV
fieldnames = list(results_row.keys())
with results_csv_path.open("w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerow(results_row)

print(f"\nResults CSV written to: {results_csv_path}")

In [ ]:
"""Cell 16: Validate results against CSV ground truth."""

print("Validation Results (vs CSV ground truth)")
print("=" * 60)
all_pass = True

gt_entry = ground_truth.get(image_filename)
if gt_entry is None:
    print(f"[WARN] No ground truth for image '{image_filename}'")
    print(f"  Available images: {list(ground_truth.keys())}")
    all_pass = False
else:
    # Compare each field
    compare_fields = [
        "DOCUMENT_TYPE",
        "RECEIPT_DATE",
        "RECEIPT_DESCRIPTION",
        "RECEIPT_TOTAL",
        "BANK_TRANSACTION_DATE",
        "BANK_TRANSACTION_DESCRIPTION",
        "BANK_TRANSACTION_DEBIT",
    ]
    for field in compare_fields:
        expected = gt_entry.get(field, "MISSING_IN_GT")
        actual = results_row.get(field, "MISSING_IN_RESULTS")
        match = str(actual).strip().upper() == str(expected).strip().upper()
        status = "PASS" if match else "FAIL"
        if not match:
            all_pass = False
        print(f"  [{status}] {field}:")
        print(f"    expected: {expected}")
        print(f"    got:      {actual}")

print("=" * 60)
if all_pass:
    print("ALL FIELDS MATCH — experiment passed.")
else:
    print("SOME FIELDS DID NOT MATCH — review above.")

In [ ]:
"""Cell 17: Comparison summary — single-pass vs 3-stage approach."""

# Tile counts
stage1_tiles = receipt_pv.shape[0]
stage2_tiles = statement_pv.shape[0]
staged_total_tiles = stage1_tiles + stage2_tiles

print("Approach Comparison: Single-Pass vs 3-Stage")
print("=" * 60)
print()
print("                    Single-Pass    3-Stage")
print("  Inference calls:  1              3 (1 + 2a + 2b)")
print(f"  Stage 1 tiles:    \u2014              {stage1_tiles} (receipt only)")
print(
    f"  Stage 2a tiles:   \u2014              {stage2_tiles} (statement \u2014 columns)"
)
print(
    f"  Stage 2b tiles:   \u2014              {stage2_tiles} (statement \u2014 matching)"
)
print(f"  Max tiles/call:   {staged_total_tiles:<14} {max(stage1_tiles, stage2_tiles)}")
print(f"  Total tiles:      {staged_total_tiles:<14} {stage1_tiles + stage2_tiles * 2}")
print()
print("Key differences:")
print(
    f"  - Single-pass: {staged_total_tiles} tiles in ONE call (shared resolution budget)"
)
print(
    f"  - 3-Stage: max {max(stage1_tiles, stage2_tiles)} tiles per call (full resolution per image)"
)
print("  - Stage 1 is inspectable before Stage 2 runs")
print("  - Stage 2a dynamically discovers column layout \u2014 works across banks")
print("  - Stage 2b uses TEXT from receipts + mapped column names, not pixels")
print()
print(f"Stage 1 receipts found:  {len(stage1_receipts)}")
print(f"Stage 2a column headers: {len(headers)} columns")
print(
    f"  Mapped: date='{date_column}', description='{description_column}', debit='{debit_column}'"
)
print(f"Stage 2b matches found:  {len(parsed_receipts)}")
print(f"Ground truth images:     {len(ground_truth)}")
print(f"Validation:              {'PASS' if all_pass else 'FAIL'}")

In [ ]:
"""Cell 18: GPU cleanup."""

import gc

try:
    model_ctx.__exit__(None, None, None)
except Exception:
    pass

del receipt_pv, statement_pv
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(f"GPU memory allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
print("Cleanup complete.")